<a href="https://colab.research.google.com/github/alimor37/WAAM_Weld_ML/blob/main/2_data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import pandas as pd
import numpy as np

# ==========================================
# STEP 1: Load cleaned dataset
# ==========================================

github_url = (
    "https://raw.githubusercontent.com/"
    "alimor37/WAAM_Weld_ML/main/"
    "Cleaned_data_numeric.xlsx"
)

df = pd.read_excel(github_url)

print("Dataset shape:", df.shape)


# ==========================================
# STEP 2: Encode Base Material
# ==========================================

print("\nBASE MATERIAL DISTRIBUTION")
print(df["Base_Material"].value_counts())

# Define binary variable
base_mapping = {
    "Weldox 700": 0,
    "Weldox 1100": 1
}

df["Base1100"] = (
    df["Base_Material"]
    .astype("string")
    .str.strip()
    .map(base_mapping)
)

# Check for unexpected categories
assert df["Base1100"].notna().all(), (
    "Unknown Base_Material category found!"
)

print("\nBase1100 encoding:")
print(
    df[["Base_Material", "Base1100"]]
    .drop_duplicates()
    .to_string(index=False)
)


# ==========================================
# STEP 3: Check Welding Methods
# ==========================================

df["Weld_Method"] = (
    df["Weld_Method"]
    .astype("string")
    .str.strip()
)

print("\nWELDING METHOD DISTRIBUTION")
print(df["Weld_Method"].value_counts())

# We keep the categorical column unchanged.
# One-Hot Encoding will be fitted inside
# the cross-validation pipeline later.


# ==========================================
# STEP 4: Define Compact Feature Set
# ==========================================

compact_numeric = [
    "Pcm",
    "t85_s",
    "No_of_Beads",
    "Dilution_Ni_pct",
    "Base1100"
]

categorical_features = [
    "Weld_Method"
]

compact_features = (
    compact_numeric + categorical_features
)


# ==========================================
# STEP 5: Define Full Chemistry Feature Set
# ==========================================

chemistry_features = [
    "C", "Si", "Mn", "P", "S",
    "Cr", "Ni", "Mo", "V", "Nb",
    "Cu", "Al", "Ti", "B", "O", "N"
]

process_features = [
    "t85_s",
    "No_of_Beads",
    "Dilution_Ni_pct",
    "Base1100"
]

full_numeric = (
    chemistry_features + process_features
)

full_features = (
    full_numeric + categorical_features
)


# ==========================================
# STEP 6: Check Missing Values
# ==========================================

print("\n" + "=" * 60)
print("COMPACT FEATURE SET")
print("=" * 60)

print("Number of original features:",
      len(compact_features))

print("\nMissing values:")
print(
    df[compact_features]
    .isna()
    .sum()
    .to_string()
)

print("\n" + "=" * 60)
print("FULL CHEMISTRY FEATURE SET")
print("=" * 60)

print("Number of original features:",
      len(full_features))

print("\nMissing values:")
print(
    df[full_features]
    .isna()
    .sum()
    .to_string()
)


# ==========================================
# STEP 7: Check Target Availability
# ==========================================

targets = [
    "Rp0.2_Mpa",
    "Rm_Mpa",
    "Cv -40ºC (J)"
]

print("\n" + "=" * 60)
print("TARGET AVAILABILITY")
print("=" * 60)

for target in targets:
    valid_n = df[target].notna().sum()

    print(f"{target}: {valid_n} valid samples")


# ==========================================
# STEP 8: Verify Feature Sets
# ==========================================

assert len(df) == 30, "Unexpected number of rows!"

assert len(compact_features) == len(
    set(compact_features)
), "Duplicate feature in Compact set!"

assert len(full_features) == len(
    set(full_features)
), "Duplicate feature in Full set!"

assert "Pcm" not in full_features

print("\n" + "=" * 60)
print("FEATURE PREPARATION COMPLETED")
print("=" * 60)

print("Compact features:", compact_features)
print("\nFull Chemistry features:", full_features)

print("\nNo rows removed.")
print("No missing values imputed.")
print("No ML model trained.")


Dataset shape: (30, 37)

BASE MATERIAL DISTRIBUTION
Base_Material
Weldox 700     16
Weldox 1100    14
Name: count, dtype: int64

Base1100 encoding:
Base_Material  Base1100
   Weldox 700         0
  Weldox 1100         1

WELDING METHOD DISTRIBUTION
Weld_Method
MAG             16
MMA              6
SAW              4
Rapid Arc        2
Laser-Hybrid     2
Name: count, dtype: Int64

COMPACT FEATURE SET
Number of original features: 6

Missing values:
Pcm                0
t85_s              0
No_of_Beads        0
Dilution_Ni_pct    0
Base1100           0
Weld_Method        0

FULL CHEMISTRY FEATURE SET
Number of original features: 21

Missing values:
C                  0
Si                 0
Mn                 0
P                  0
S                  0
Cr                 0
Ni                 0
Mo                 0
V                  0
Nb                 0
Cu                 0
Al                 0
Ti                 0
B                  0
O                  6
N                  0
t85_s     

In [2]:

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

# ==========================================
# STEP 1: Load cleaned dataset
# ==========================================

github_url = (
    "https://raw.githubusercontent.com/"
    "alimor37/WAAM_Weld_ML/main/"
    "Cleaned_data_numeric.xlsx"
)

df = pd.read_excel(github_url)

# ==========================================
# STEP 2: Prepare categorical columns
# ==========================================

df["Base_Material"] = (
    df["Base_Material"]
    .astype("string")
    .str.strip()
)

base_mapping = {
    "Weldox 700": 0,
    "Weldox 1100": 1
}

df["Base1100"] = (
    df["Base_Material"].map(base_mapping)
)

if df["Base1100"].isna().any():
    raise ValueError("Unknown Base_Material category.")

df["Weld_Method"] = (
    df["Weld_Method"]
    .astype("string")
    .str.strip()
)

if df["Weld_Method"].isna().any():
    raise ValueError("Missing Weld_Method values found.")

# ==========================================
# STEP 3: Define feature sets
# ==========================================

compact_numeric = [
    "Pcm",
    "t85_s",
    "No_of_Beads",
    "Dilution_Ni_pct",
    "Base1100"
]

chemistry_features = [
    "C", "Si", "Mn", "P", "S",
    "Cr", "Ni", "Mo", "V", "Nb",
    "Cu", "Al", "Ti", "B", "O", "N"
]

full_numeric = chemistry_features + [
    "t85_s",
    "No_of_Beads",
    "Dilution_Ni_pct",
    "Base1100"
]

categorical_features = ["Weld_Method"]

compact_features = (
    compact_numeric + categorical_features
)

full_features = (
    full_numeric + categorical_features
)

# ==========================================
# STEP 4: Numeric preprocessing
# ==========================================

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

# ==========================================
# STEP 5: Categorical preprocessing
# ==========================================

categorical_transformer = Pipeline(
    steps=[
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

# ==========================================
# STEP 6: Compact preprocessing
# ==========================================

compact_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            compact_numeric
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

# ==========================================
# STEP 7: Full Chemistry preprocessing
# ==========================================

full_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            full_numeric
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

# ==========================================
# STEP 8: Verify setup
# ==========================================

print("Dataset shape:", df.shape)

print("\nCOMPACT MODEL")
print("Numeric features:", len(compact_numeric))
print("Categorical features:", categorical_features)

print("\nFULL CHEMISTRY MODEL")
print("Numeric features:", len(full_numeric))
print("Categorical features:", categorical_features)

print("\nMissing oxygen values:",
      df["O"].isna().sum())

print("\nPreprocessing pipelines created.")
print("No model trained.")
print("No rows deleted.")
print("No missing values replaced yet.")


Dataset shape: (30, 38)

COMPACT MODEL
Numeric features: 5
Categorical features: ['Weld_Method']

FULL CHEMISTRY MODEL
Numeric features: 20
Categorical features: ['Weld_Method']

Missing oxygen values: 6

Preprocessing pipelines created.
No model trained.
No rows deleted.
No missing values replaced yet.
